![ATARRI logos](img/logos.png)

# 2.3 Interactive plotting tools in Jupyter

#### Objective 

The objective of this notebook is to provide techniques to create quick interactive visualisations of MONARCH dust forecast datasets.

## Why are interactive visualisations useful?

Thus far we have covered how to create map visualisations of our dust forecasts at static time steps and statistical overviews. However, making use of interactive visualisations allows us to visualise the spatio-temporal evolution of our dust forecasts. This can be especially useful to observe  and assess any changes in plumes, possible affected areas, magnitude of the event, etc.

For this notebook, we will make use of two interactive functionalities available to us:

- Jupyter notebook widgets
- GIF files

We will start by importing our libraries. We will also need our `functions1.ipynb` file, since we will make use of some custom functions.

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as widgets
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import BoundaryNorm
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, clear_output
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
from IPython.utils.io import capture_output
with capture_output():
    %run ../functions/functions1.ipynb

## Exercise 1: Widget slider

First, we will use `ipywidgets` to create a slider with all the time steps that cover our dust event. Our goal will be to drag the slider so that we see different time steps form the `ds` dataset interactively. We will begin by loading and merging our datasets.

In [3]:
ds = xr.open_mfdataset(
    "/shared/data/exp_to_interp/monarch/regional/3hourly/od550_dust/*.nc",
    preprocess=select_timesteps,
    combine='by_coords'
)
ds

<xarray.Dataset> Size: 174MB
Dimensions:     (time: 32, lat: 825, lon: 1650)
Coordinates:
  * time        (time) datetime64[ns] 256B 2025-03-04 ... 2025-03-07T21:00:00
  * lat         (lat) float64 7kB -10.95 -10.85 -10.75 ... 71.25 71.35 71.45
  * lon         (lon) float64 13kB -62.95 -62.85 -62.75 ... 101.8 101.9 102.0
Data variables:
    od550_dust  (time, lat, lon) float32 174MB dask.array<chunksize=(1, 825, 1650), meta=np.ndarray>
Attributes:
    Domain:                     Regional
    Conventions:                CF-1.7
    comment:                    Generated on cirrus
    history_of_appended_files:  Mon Mar  3 18:03:43 2025: Appended file od550...
    NCO:                        netCDF Operators version 4.8.1 (Homepage = ht...
    history:                    Tue Mar  4 11:34:48 2025: ncap2 -O -v -s wher...

To create this widget, we need to start by creating the actual slider. We will first extract the time steps themselves:

In [4]:
n_times = ds.dims["time"]
n_times

32

Next, we will use the `widgets.IntSlider` function. It will require the following inputs:

- `value`: initial value of the slider.
- `min`: minimum value.
- `max`: maximum value.
- `step`: amount that the slider can move.
- `description`: Label next to the slider.

In [5]:
time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=n_times - 1,
    step=1,
    description="Time"
)

To help us plot our dust variable at each time step, we will use the custom function `plot_variable_time_step()`. Our custom function works by clearing previous output, extracting data from our pre-defined dust variable at a specific time step, and creating our plot structure with the input parameters. To start, we will define the following:

- `var_name`: The name of the variable we are going to plot.
- `cmap`: The colourmap. We will stick to the BDRC colour scheme.
- `norm`: The normalised ranges for grouping our data. We will also stick to the BDRC bins.

In [6]:
var_name = list(ds.data_vars)[0] # we directly extract the name of the variable from the dataset
cmap = get_cmap_bdrc()
norm = get_normalised_bdrc(var_name, cmap)

In [7]:
var_name

'od550_dust'

Once the slider widget is ready, we can create the interactive plots with `interact`. This is a helper function that automatically calls our custom `plot_variable_time_step` function whenever one of the slider values changes, and then passes those values as arguments to the function.

To simplify the interactive change of time steps, we can use the `lambda` shortcut. By setting `time_index=time_slider`, we will make the `interact` function call the updated `time_index` into the custom plotting function every time the slider moves.

In [8]:
widgets.interact(
    lambda time_index: plot_variable_time_step(time_index, ds, var_name, cmap, norm),
    time_index=time_slider
);

interactive(children=(IntSlider(value=0, description='Time', max=31), Output()), _dom_classes=('widget-interac…

## Exercise 2: GIF files

The downside of using the widget functionality is that they can only be run inside interactive Jupyter environments. Furthermore, if we shut down our kernel, the widget closes and we need to rerun our notebook to refresh the output.

As an alternative, we will work to produce a GIF of our MONARCH dataset. The advantage to this is that we can output it to our `/plot_outputs/` directory, and access this file from outside the Jupyter interface.

In order to achieve this, we need to initialise and plot the first time step of the dataset, after which we will update the plotting object with the data from each subsequent time.

First, we will set our working objects:
- latitude
- longitude
- the number of frames (i.e. the number of time steps from `ds`)
- our dust variable
- DOD data from the initial time step
- the title we want to give to our plot

In [9]:
lat = ds['lat']
lon = ds['lon']
time = ds['time']
n_frames = len(time)
ds_dod = ds["od550_dust"]
ds_dod_init = ds_dod.isel(time=0)
gif_title = "MONARCH DOD at time"

Next, we will create our standard plotting setup, with DOD at the first time step.

Following this, we will create our animation with `FuncAnimation`, through which we can create animations by repeatedly calling a function that updates parts of a plot. For this last part, we will use the custom `update` function, which is going to take a slice of data for each new time. We can combine this with the following parameters to create the gif:

- `fig`: The figure object we created before.
- `update`: The function to call for each frame of the animation.
- `frames=n_frames`: Establish the number of frames that our animation should have, which in this case should be the same as the number of time steps.
- `fargs=(mesh,gif_title)`: Additional arguments that will be passed to the `update` function. In our case, we will pass the `mesh` object and the `gif_title`.
- `blit=False`: Optimisation parameter that redraws the figure each frame.

Finally, we save our animation as a GIF, setting `fps=2` for a display of 2 frames per second.

In [10]:
fig, ax = plt.subplots(figsize=(14,8), subplot_kw={'projection': ccrs.PlateCarree()})

# Initial plot
mesh = ax.pcolormesh(lon,
                     lat,
                     ds_dod_init,
                     transform=ccrs.PlateCarree(),
                     cmap=cmap,
                     norm=norm)
cbar = plt.colorbar(mesh, ax=ax, orientation="vertical", label=var_name)
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':') 
title = ax.set_title(f'{gif_title} {str(time[0].values)}') # Print the title with our input and the initial time

# Create animation
anim = FuncAnimation(fig,
                     update,
                     frames=n_frames, # number of frames equals number of time steps
                     fargs=(mesh,gif_title), # set the mesh object and the gif title as input arguments of the update function
                     blit=False)
anim.save("../plot_outputs/od550_dust_animation.gif", writer=PillowWriter(fps=2))
plt.close(fig)

## References and further reading

- Ipywidgets [documentation](https://github.com/jupyter-widgets/ipywidgets#readme)
- Jupyter Widgets 8.1.7 [documentation](https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20List.html)
- Matplotlib `.animation.FuncAnimation` [documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FuncAnimation.html)
- Matplotlib `.pyplot.pcolormesh` [documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.pcolormesh.html)

<table style="width:100%;">
    <tr>
        <td style="text-align: center;"><a href="VT2-2-combining_MONARCH_datasets.ipynb" style="font-size: 18px;">⬅ Previous</a></td>
        <td style="text-align: center;"><a href="VT2-4-vertical_profiles.ipynb" style="font-size: 18px;">Next ➡</a></td>
    </tr
</table>